# 5 · Fine-tuning Wav2Vec2 XLS-R-300M

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/05_finetune/05_wav2vec2_finetune.ipynb)

**Pipeline stage 5 of 6.** Fine-tune `facebook/wav2vec2-xls-r-300m` for Kölsch
phoneme recognition with a CTC head. Transfer learning from a model pre-trained
on 436k h / 128 languages means a few hours of Kölsch suffice for adaptation.

> Needs a GPU (Colab → Runtime → Change runtime type → GPU).

## Setup

In [ ]:
!pip -q install "transformers>=4.40" datasets evaluate jiwer torchaudio accelerate
import torch, json, numpy as np
from dataclasses import dataclass
from typing import Dict, List, Union
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "·", device)

## 1 · Load the manifest (audio + phoneme labels)

Join the segment manifest from **Notebook 3** with the `phonetic` labels from
**Notebook 4**. Each row needs an `audio_path` and a `phonetic` string
(`p h | p h`). Then make a speaker-disjoint train/valid/test split.

In [ ]:
from datasets import Dataset, Audio
import pandas as pd

# df = pd.read_csv("manifest.csv")   # columns: audio_path, phonetic, speaker_id, split
# Demo stub so the cell runs; replace with your manifest:
df = pd.DataFrame({"audio_path":[], "phonetic":[], "split":[]})

def to_ds(split):
    sub = df[df.split==split]
    ds = Dataset.from_pandas(sub[["audio_path","phonetic"]], preserve_index=False)
    return ds.cast_column("audio_path", Audio(sampling_rate=16000))

# train_ds, valid_ds, test_ds = to_ds("train"), to_ds("valid"), to_ds("test")
print("Plug in manifest.csv (audio_path, phonetic, split) to build the datasets.")

## 2 · Build the phoneme vocabulary + processor

In [ ]:
from transformers import (Wav2Vec2PhonemeCTCTokenizer, Wav2Vec2FeatureExtractor,
                          Wav2Vec2Processor)

def build_vocab(phonetic_series, path="vocab.json"):
    toks = set()
    for s in phonetic_series:
        toks.update(s.split())          # phonemes and the '|' delimiter
    toks.discard("|")
    vocab = {t: i for i, t in enumerate(sorted(toks))}
    vocab["|"]   = len(vocab)            # word delimiter
    vocab["[UNK]"] = len(vocab)
    vocab["[PAD]"] = len(vocab)
    json.dump(vocab, open(path,"w"), ensure_ascii=False)
    return vocab

# vocab = build_vocab(df[df.split=="train"]["phonetic"])
# The labels are ALREADY phonemes (Notebook 4), so we use the PHONEME tokenizer
# with do_phonemize=False — it maps IPA phoneme tokens (space-separated) to ids
# and treats '|' as the word delimiter. This is phoneme recognition, not word ASR.
# tokenizer = Wav2Vec2PhonemeCTCTokenizer("vocab.json", unk_token="[UNK]",
#                 pad_token="[PAD]", word_delimiter_token="|",
#                 phone_delimiter_token=" ", do_phonemize=False)
# fe = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0,
#                 do_normalize=True, return_attention_mask=True)
# processor = Wav2Vec2Processor(feature_extractor=fe, tokenizer=tokenizer)
print("vocab builder ready (44 phones + | + [UNK] + [PAD] ≈ 47 tokens)")

## 3 · Prepare dataset (audio → input_values, phonetic → labels)

In [ ]:
def prepare(batch):
    audio = batch["audio_path"]
    batch["input_values"] = processor(audio["array"],
                                      sampling_rate=16000).input_values[0]
    batch["labels"] = processor(text=batch["phonetic"]).input_ids
    return batch

# train_ds = train_ds.map(prepare, remove_columns=train_ds.column_names)
# valid_ds = valid_ds.map(prepare, remove_columns=valid_ds.column_names)

## 4 · CTC data collator

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features):
        inp = [{"input_values": f["input_values"]} for f in features]
        lab = [{"input_ids":   f["labels"]}        for f in features]
        batch = self.processor.pad(inp, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            lb = self.processor.pad(lab, padding=self.padding, return_tensors="pt")
        batch["labels"] = lb["input_ids"].masked_fill(lb.attention_mask.ne(1), -100)
        return batch
# data_collator = DataCollatorCTCWithPadding(processor=processor)

## 5 · Metrics — WER and CER (the metrics logged during training)

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    logits = pred.predictions
    ids = np.argmax(logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str),
            "cer": cer_metric.compute(predictions=pred_str, references=label_str)}

## 6 · Model + training configuration

In [ ]:
from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer

# model = Wav2Vec2ForCTC.from_pretrained(
#     "facebook/wav2vec2-xls-r-300m",
#     attention_dropout=0.05, hidden_dropout=0.05, feat_proj_dropout=0.05,
#     mask_time_prob=0.05, layerdrop=0.05, ctc_loss_reduction="mean",
#     ctc_zero_infinity=True,
#     pad_token_id=processor.tokenizer.pad_token_id,
#     vocab_size=len(processor.tokenizer))
# model.freeze_feature_encoder()
# model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir="kolsch_wav2vec2_model",
    group_by_length=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,        # effective batch 16
    per_device_eval_batch_size=2,
    num_train_epochs=150,
    fp16=True,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=500,
    weight_decay=0.05,
    eval_strategy="steps", eval_steps=1000,
    save_strategy="steps", save_steps=1000,
    logging_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="wer",          # select best checkpoint on validation WER
    greater_is_better=False,
    save_total_limit=2,
)
print("training args ready")

## 7 · Train, then save model + processor

In [ ]:
# trainer = Trainer(
#     model=model, args=args, data_collator=data_collator,
#     train_dataset=train_ds, eval_dataset=valid_ds,
#     compute_metrics=compute_metrics, tokenizer=processor.feature_extractor)
# trainer.train()
# trainer.save_model("kolsch_wav2vec2_model")
# processor.save_pretrained("kolsch_wav2vec2_model")
print("uncomment to train (GPU). Best checkpoint ≈ 15.2% val WER / 11.8% val CER.")

## Result

Selecting the best checkpoint on **validation WER** (not loss — CTC validation
loss rebounds) gives ~15.2% val WER / 11.8% val CER, generalising to
**14.63% WER / 11.75% CER** on the held-out test set (see Notebook 6).